In [1]:
import pandas as pd
from datasets import load_dataset

# Load content table
content = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    split="train"
)

content_df = content.to_pandas()

print(content_df.shape)
content_df.head()

(519606, 26)


,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682.0,2555.0,None,None,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438.0,2430.0,None,None,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576.0,2645.0,None,None,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457.0,2522.0,None,None,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776.0,2552.0,None,None,True,False


# 1. Two Paper Findings + My Methodology Questions

## Finding 1: Content refreshing can improve search performance.

**Methodology Question:**
How was the improvement measured? Was it compared against a control group of content that was not refreshed, and was the evaluation performed using a time-aware validation strategy?

---

## Finding 2: AI-generated content can support scalable content production.

**Methodology Question:**
How was content quality evaluated? Were search performance improvements measured over a sufficient time period, and were differences across industries or content types considered?

In [2]:
from sklearn.model_selection import GroupShuffleSplit

# Select simple features
features = content_df[
    [
        "search_volume",
        "competition",
        "word_count",
        "char_count",
        "backlinks",
    ]
].fillna(0)

# Simple baseline target
target = (content_df["is_published"] == True).astype(int)

groups = content_df["client_hash_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(gss.split(features, target, groups))

X_train = features.iloc[train_idx]
X_test = features.iloc[test_idx]

y_train = target.iloc[train_idx]
y_test = target.iloc[test_idx]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Unique train clients:", groups.iloc[train_idx].nunique())
print("Unique test clients:", groups.iloc[test_idx].nunique())

Training samples: 439038
Testing samples: 80568
Unique train clients: 67
Unique test clients: 17


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

features = content_df[
    [
        "search_volume",
        "competition",
        "word_count",
        "char_count",
        "backlinks",
    ]
].fillna(0)

target = (content_df["is_published"] == True).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.2,
    random_state=42
)

model_random = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model_random.fit(X_train, y_train)

pred_random = model_random.predict(X_test)

random_accuracy = accuracy_score(y_test, pred_random)

print("Random Split Accuracy:", random_accuracy)

Random Split Accuracy: 0.8120032331941264


In [4]:
from sklearn.model_selection import GroupShuffleSplit

groups = content_df["client_hash_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(gss.split(features, target, groups))

X_train = features.iloc[train_idx]
X_test = features.iloc[test_idx]

y_train = target.iloc[train_idx]
y_test = target.iloc[test_idx]

model_group = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model_group.fit(X_train, y_train)

pred_group = model_group.predict(X_test)

group_accuracy = accuracy_score(y_test, pred_group)

print("Grouped Split Accuracy:", group_accuracy)

Grouped Split Accuracy: 0.712069307913812


## Validation Comparison

The grouped validation split prevents information leakage between clients and provides a more realistic estimate of model performance. The grouped split is considered a more honest evaluation than a random split because data from the same client does not appear in both training and testing sets.

In [5]:
leakage_features = [
    "search_volume",
    "competition",
    "word_count",
    "char_count",
    "backlinks"
]

print("Features Used:")
for feature in leakage_features:
    print("-", feature)

Features Used:
- search_volume
- competition
- word_count
- char_count
- backlinks


# Leakage Audit

The selected features are available before prediction and do not use future performance metrics.

No future clicks, impressions, sessions, rankings, or manually created labels were used during model training.

No target-derived information was included in the feature set.

In [6]:
results = X_test.copy()

results["Actual"] = y_test.values
results["Prediction"] = pred_group

errors = results[
    results["Actual"] != results["Prediction"]
]

errors.head(10)

,search_volume,competition,word_count,char_count,backlinks,Actual,Prediction
272,390.0,1.00,2628.0,15854.0,180.0,0,1
281,390.0,0.87,2393.0,14386.0,63.0,0,1
321,0.0,0.00,1900.0,11320.0,0.0,1,0
740,880.0,1.00,2543.0,15474.0,241.0,0,1
973,110.0,1.00,2288.0,14698.0,98.0,0,1
1023,10.0,0.00,2554.0,16275.0,0.0,1,0
1074,4400.0,0.17,2316.0,14690.0,172.0,0,1
7137,0.0,0.00,0.0,0.0,0.0,1,0
7138,0.0,0.00,0.0,0.0,0.0,1,0
7140,20.0,0.02,3133.0,22574.0,11.0,1,0


## Error Review

Possible reasons for incorrect predictions include:

- Limited feature set.
- Missing contextual information.
- Industry-specific behavior.
- Search intent differences.
- Content quality not represented by structured features.

# Claim Rewrite

Original Claim

"The model accurately predicts which content should be optimized."

Rewritten Claim

"The model observed relationships between historical content features and publishing status. The results provide decision-support signals rather than definitive recommendations."

---

Original Claim

"Our model identifies the best pages to refresh."

Rewritten Claim

"The baseline model ranks pages using measurable historical signals and should be used as one input into editorial decision making."

---

Original Claim

"The model improves SEO performance."

Rewritten Claim

"The observed feature relationships may help prioritize content reviews, but additional validation is required before claiming improvements in SEO performance."

# Self Check

✅ Two research findings reviewed.

✅ Methodology questions documented.

✅ Random split evaluated.

✅ Honest grouped split evaluated.

✅ Before vs after comparison completed.

✅ Leakage audit completed.

✅ Error examples reviewed.

✅ Claims rewritten using safe language.

✅ Notebook executed successfully.